# Topics of Practice: Data Bigger Than Your Laptop

**GSB 5544 · Computing and Machine Learning for Business Analytics**

This notebook walks through the main functions from the pre-class reading, one short block at a time. Each block names the section of the reading it draws on. Replace every `____` with the right code and run the cell.

You need: `psutil`, `boto3`, `s3fs`, `pandas`, `matplotlib`. No AWS account is required.

In [ ]:
import io
import psutil
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "noaa-ghcn-pds"

## A. What your machine actually has

*From the reading, §1.* Total RAM is not available RAM. The available number decides whether a file loads.

Fill in the two calls that report memory and the attribute that gives the memory available **right now**.

In [ ]:
vm = psutil.virtual_memory()
du = psutil.disk_usage("/")

print(f"Physical cores : {psutil.cpu_count(logical=False)}")
print(f"Total RAM      : {vm.total / 1e9:.1f} GB")
print(f"Available RAM  : {vm.available / 1e9:.1f} GB")
print(f"Free disk      : {du.free / 1e9:.1f} GB")

MY_RAM = vm.available          # keep this; we compare file sizes against it below

## B. The units of data

*From the reading, §2.* Byte counts are unreadable past a few million. Write the helper that turns them into KB / MB / GB and so on, using base 1000.

Fill in the list of units in ascending order and the line that steps `n` down one unit.

In [ ]:
def human(n, base=1000):
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if n < base:
            return f"{n:,.1f} {unit}"
        n /= base
    return f"{n:,.1f} EB"

print(human(1_500_000))          # expect 1.5 MB
print(human(1_336_457_184))      # expect 1.3 GB
print(human(MY_RAM))             # your available RAM, readable

**Check yourself.** A "1 TB" drive shows as 931 GB in your file browser. Which base does the drive maker use, and which does the operating system use? Write your answer in the cell below as a comment.

In [ ]:
# Drive maker uses base 1000; the OS uses base 1024.
# 1e12 / 1024**3 = 931.3 GiB, which the OS labels "GB".

## C. The vocabulary of AWS

*From the reading, §3.* Fill in the term that each definition describes. The cell checks your answers.

In [ ]:
vocab = {
    "A named container for objects; its name is globally unique":     "bucket",
    "One stored item plus its metadata such as size":                  "object",
    "The full name of an object inside its bucket":                    "key",
    "The start of a key, used to list everything 'under' something":   "prefix",
    "A geographic cluster of data centers such as us-east-1":          "region",
    "A rented virtual computer":                                       "EC2",
    "The service that decides who may do what":                        "IAM",
}

expected = {"bucket", "object", "key", "prefix", "region", "ec2", "iam"}
given = {v.strip().lower() for v in vocab.values()}
print("All correct" if given == expected else f"Check: {expected - given}")

## D. Connect without an account

*From the reading, §4.* Public datasets on S3 can be read with an **unsigned** request. Build the client, then list the top-level "folders" of the GHCN bucket.

Fill in the signature setting that makes the request anonymous, and the delimiter that makes S3 group keys like folders.

In [ ]:
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),
)

resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")

print("Prefixes :", [p["Prefix"] for p in resp.get("CommonPrefixes", [])])
print("Files    :", [o["Key"] for o in resp.get("Contents", [])][:6])

Look at the output. `csv/` is not a folder. In one sentence, what is it? Type your answer as a comment.

In [ ]:
# It is a prefix: the shared beginning of many object keys. S3 groups keys by it when asked, which looks like a folder.

## E. Read a size without reading the file

*From the reading, §4.* Every object carries its size as metadata. List the by-year files for the 2020s and print each size in readable units.

Fill in the prefix that selects `csv/by_year/2020.csv` through `2026.csv`, and the field that holds an object's size.

In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="csv/by_year/202")

sizes = {}
for obj in resp["Contents"]:
    sizes[obj["Key"]] = obj["Size"]
    print(f"{obj['Key']:<28} {human(obj['Size'])}")

Now the comparison that the whole reading builds toward. pandas typically needs about **3×** a text CSV's size in memory. For each file, decide whether a plain `pd.read_csv` would fit in your available RAM.

Fill in the multiplier and the comparison.

In [ ]:
MULT = 3

for key, size in sizes.items():
    need = size * MULT
    fits = need < MY_RAM
    print(f"{key:<28} needs ~{human(need):>9}   fits: {fits}")

## F. Peek at a file with a range request

*From the reading, §4.* You can read the first few hundred bytes of a gigabyte file to see its shape. Do that for the 2024 by-year file and check whether it has a header row.

Fill in the `Range` header for the first 300 bytes.

In [ ]:
key = "csv/by_year/2024.csv"

obj = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-299")
first = obj["Body"].read().decode()
print(first)

has_header = first.upper().startswith("ID,")
print("Header row present:", has_header)

## G. Extract one station, not one year

*From the reading, §5.* Lever 1 is picking the right object. The Cal Poly station (`USC00047851`, SAN LUIS OBISPO POLY) has records since 1893 in a single file of about 7 MB. Read it, keeping only the four columns you need.

Fill in the station key, the header logic, and the `usecols` list.

In [ ]:
COLS = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
STATION = "USC00047851"
key = f"csv/by_station/{STATION}.csv"

# size first, always
print("Size:", human(s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]))

# header check
first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
has_header = first.upper().startswith("ID,")

raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
df = pd.read_csv(
    io.BytesIO(raw),
    header=0 if has_header else None,
    names=COLS,
    usecols=["id", "date", "element", "value"],
    dtype={"id": "string", "element": "string"},
)

print(df.shape)
print("In memory:", human(df.memory_usage(deep=True).sum()))
df.head()

## H. Transform

*From the reading, §5, "The units trap".* Dates are stored as `YYYYMMDD` integers and temperatures in tenths of a degree Celsius. Fix both, keep only daily maximum temperature, and compute a yearly mean.

Fill in the date format string, the element code, the unit conversion, and the resampling rule for year-end.

In [ ]:
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")

tmax = df[df["element"] == "TMAX"].copy()
tmax["tmax_c"] = tmax["value"] / 10

yearly = (
    tmax.set_index("date")["tmax_c"]
        .resample("YE")          # "Y" on pandas < 2.2
        .mean()
)

print(f"{len(tmax):,} TMAX readings, {yearly.index.min().year}–{yearly.index.max().year}")
yearly.tail()

## I. Bring the visual

*From the reading, §6.* One picture that came from a bucket you never downloaded.

Fill in the series to plot and the y-axis label with its unit.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(yearly.index, yearly.values, linewidth=1.2)
ax.set_ylabel("Mean daily maximum (°C)")
ax.set_title(f"{STATION}: yearly mean of daily maximum temperature")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Before you leave

Write one sentence, as a comment, that answers: *why did reading the station file work when reading the year file would not have?*

In [ ]:
# The station file holds only the rows I needed, about 7 MB. The year file holds every station on Earth, 1.3 GB, and I would have thrown almost all of it away. Choosing the object was the filter.